# Tech Challenge Fase 2
## 04.4 — Gold Streaming

Consolida os eventos válidos em uma visão analítica atual por município e indicador.

## 1. Imports e configuração

In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"
config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))
STREAMING_PATH = config["paths"]["streaming_path"]

SILVER_PATH = f"{STREAMING_PATH}/silver_streaming"
GOLD_CURRENT = f"{STREAMING_PATH}/gold_streaming/current"
GOLD_HISTORY = f"{STREAMING_PATH}/gold_streaming/history"

## 2. Construção dos indicadores

In [0]:
df_silver = spark.read.parquet(SILVER_PATH)

chaves = ["ano","co_uf","sg_uf","co_municipio","no_municipio","indicador"]

df_agregado = (
    df_silver.groupBy(chaves)
    .agg(
        F.avg("valor").alias("valor_medio"),
        F.max("event_timestamp").alias("ultima_atualizacao"),
        F.count("*").alias("quantidade_eventos")
    )
)

w = Window.partitionBy(*chaves).orderBy(F.col("event_timestamp").desc(), F.col("event_version").desc())

df_ultimo = (
    df_silver
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn")==1)
    .select(*chaves, F.col("valor").alias("ultimo_valor"), F.col("event_type").alias("ultimo_tipo_evento"))
)

df_gold = (
    df_agregado.join(df_ultimo, on=chaves, how="left")
    .withColumn("_gold_streaming_processed_at", F.current_timestamp())
)

## 3. Persistência e validação

In [0]:
(df_gold.coalesce(1).write.mode("overwrite").format("parquet").save(GOLD_CURRENT))

(
    df_gold.withColumn("snapshot_date", F.current_date())
    .write.mode("append").format("parquet")
    .partitionBy("snapshot_date")
    .save(GOLD_HISTORY)
)

df_validacao = spark.read.parquet(GOLD_CURRENT)

if df_validacao.count() == 0:
    print("Gold Streaming sem registros.")
else:
    display(df_validacao.orderBy(F.col("ultima_atualizacao").desc()))